# Alpamayo 1.5: Local Offline Inference Demo

This notebook runs Alpamayo 1.5 on a local offline clip directory, shows all sampled images used for inference, and visualizes the predicted trajectory and chain-of-thought.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import load_offline_dataset
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5

In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
T0_SOD = 175490.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('T0_SOD =', T0_SOD)

### Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')

### Load local offline data

In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
)

print('Offline dataset loaded.')
print('camera_indices:', data['camera_indices'].tolist())
print('image_frames shape:', tuple(data['image_frames'].shape))
print('ego_history_xyz shape:', tuple(data['ego_history_xyz'].shape))
print('ego_future_xyz shape:', tuple(data['ego_future_xyz'].shape))
print('requested frame indices:', data['video_frame_indices'][0].tolist())
print('actual frame indices (first camera):', data['actual_video_frame_indices'][0].tolist())

### Show all sampled images used for inference

Rows = cameras, columns = temporal frames. This includes all historical images used by the model.

In [ ]:
image_frames = data['image_frames'].cpu().numpy()  # [N_cam, T, C, H, W]
camera_indices = data['camera_indices'].cpu().tolist()
requested_indices = data['video_frame_indices'].cpu().numpy()
actual_indices = data['actual_video_frame_indices'].cpu().numpy()

num_cams, num_frames, _, _, _ = image_frames.shape
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

fig, axes = plt.subplots(num_cams, num_frames, figsize=(4 * num_frames, 3.5 * num_cams))

if num_cams == 1 and num_frames == 1:
    axes = np.array([[axes]])
elif num_cams == 1:
    axes = np.array([axes])
elif num_frames == 1:
    axes = np.array([[ax] for ax in axes])

for cam_idx in range(num_cams):
    for t_idx in range(num_frames):
        ax = axes[cam_idx, t_idx]
        frame = np.transpose(image_frames[cam_idx, t_idx], (1, 2, 0))
        ax.imshow(frame)
        ax.axis('off')
        ax.set_title(
            f'{camera_names[cam_idx]}\n'
            f't={t_idx}, req={requested_indices[cam_idx, t_idx]}, actual={actual_indices[cam_idx, t_idx]}'
        )

plt.tight_layout()
plt.show()

### Prepare chat template

In [ ]:
messages = helper.create_message(
    frames=data['image_frames'].flatten(0, 1),
    camera_indices=data['camera_indices'],
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors='pt',
)

print('seq length:', inputs.input_ids.shape)

model_inputs = {
    'tokenized_data': inputs,
    'ego_history_xyz': data['ego_history_xyz'],
    'ego_history_rot': data['ego_history_rot'],
}
model_inputs = helper.to_device(model_inputs, DEVICE)

### Run inference

In [ ]:
if DEVICE.startswith('cuda'):
    torch.cuda.manual_seed_all(42)
    autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
else:
    autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

with autocast_context:
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=model_inputs,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        return_extra=True,
    )

print('Inference done.')

### Show CoT and compute minADE

In [ ]:
cot_value = extra['cot'][0]
if hasattr(cot_value, 'tolist'):
    cot_value = cot_value.tolist()

print('Chain-of-Causation:')
print(cot_value)

gt_xyz = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]
gt_xy = gt_xyz[:, :2].T

pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]
pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)

diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
min_ade = float(diff.min())

print('minADE:', min_ade, 'meters')

### Plot predicted trajectory vs GT

In [ ]:
def _compute_adaptive_xy_limits(gt_xyz, pred_xyz_np, min_span=20.0, pad_ratio=0.12, pad_min=2.0):
    pts = [gt_xyz[:, :2], np.array([[0.0, 0.0]], dtype=np.float32)]
    for sample_idx in range(pred_xyz_np.shape[0]):
        pts.append(pred_xyz_np[sample_idx, :, :2])

    pts = np.concatenate(pts, axis=0)
    xmin, ymin = pts.min(axis=0)
    xmax, ymax = pts.max(axis=0)

    xspan = xmax - xmin
    yspan = ymax - ymin
    span = max(float(xspan), float(yspan), float(min_span))

    xcenter = 0.5 * (xmin + xmax)
    ycenter = 0.5 * (ymin + ymax)

    pad = max(span * pad_ratio, pad_min)
    half = 0.5 * span + pad

    xlim = (xcenter - half, xcenter + half)
    ylim = (ycenter - half, ycenter + half)
    return xlim, ylim


xlim, ylim = _compute_adaptive_xy_limits(
    gt_xyz=gt_xyz,
    pred_xyz_np=pred_xyz_np,
    min_span=20.0,
    pad_ratio=0.12,
    pad_min=2.0,
)

plt.figure(figsize=(6, 6))

plt.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=2, markersize=3, label='GT')

for sample_idx in range(pred_xyz_np.shape[0]):
    plt.plot(
        pred_xyz_np[sample_idx, :, 0],
        pred_xyz_np[sample_idx, :, 1],
        '-o',
        linewidth=2,
        markersize=3,
        alpha=0.8,
        label=f'Pred {sample_idx}',
    )

plt.scatter([0.0], [0.0], c='red', marker='x', s=100, label='t0 ego')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Offline inference @ t0_sod={T0_SOD}, minADE={min_ade:.3f}m')
plt.xlim(*xlim)
plt.ylim(*ylim)
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()